In [42]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

from typing import TypedDict
from dotenv import load_dotenv
import os

In [43]:
load_dotenv()
open_ai_api_key = os.environ.get('OPEN_AI_API_KEY')
google_gemini_api_key = os.environ.get('GOOGLE_GEMINI_API_KEY')

In [ ]:
# model = ChatOpenAI(api_key = open_ai_api_key)
model = ChatGoogleGenerativeAI(
    api_key=google_gemini_api_key, 
    model="gemini-3-flash-preview"
)

In [45]:
# create a state
class LLMState(TypedDict):
    question: str
    answer : str

In [46]:
def llm_qa(state : LLMState) -> LLMState:
    # extract a question
    question = state['question']
    # form a prompt
    prompt = f'answer the follwing question : {question}'
    # ask to LLM
    answer = model.invoke(prompt).content
    # update answer in the state
    state['answer'] = answer
    
    return state

In [47]:
# create a graph

graph = StateGraph(LLMState)

# Nodes
graph.add_node('llm_qa',llm_qa)
# edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa',END)

# compile
workflow = graph.compile()

In [48]:
# exec the graph
initial_state = {
    'question' : 'What is the capital of India?'
}
final_state = workflow.invoke(initial_state)

print(final_state['answer'][0]['text'])

The capital of India is **New Delhi**.
